# 🛡️ Scam Detection — LoRA Fine-Tuning Pipeline
### Model: `Qwen2.5-3B-Instruct` | Method: QLoRA (4-bit) | Trainer: TRL SFTTrainer

---
**Pipeline Overview:**
```
Audio ──► Whisper ──► Transcript ──► [This Model] ──► Structured JSON Assessment
```

**What this notebook does:**
1. Installs all dependencies
2. Loads & validates your `scam_dataset.jsonl`
3. Applies 4-bit quantization (BitsAndBytes)
4. Attaches LoRA adapters via PEFT
5. Fine-tunes with SFTTrainer (TRL)
6. Saves the adapter-only weights
7. Runs a live inference example

> ⚡ **Runtime:** Go to `Runtime → Change runtime type → T4 GPU` before running.

## ✅ Step 0 — Verify GPU

In [ ]:
# ══════════════════════════════════════════════════════
# MASTER BOOTSTRAP — Run this first on every new session
# ══════════════════════════════════════════════════════
import subprocess, sys, torch

# 1. Install all packages in one shot
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "peft>=0.13.0", "trl>=0.12.0", "accelerate>=0.34.0",
    "bitsandbytes>=0.44.0", "datasets>=2.20.0",
    "sentencepiece", "protobuf", "einops"
])

# 2. Immediately load model so it's in memory for all steps below
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_storage=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, trust_remote_code=True, padding_side="right"
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    dtype=torch.bfloat16,
)
model.config.use_cache = False
model.config.pretraining_tp = 1

print(f"✅ Ready — GPU: {torch.cuda.get_device_name(0)}")
print(f"✅ VRAM : {torch.cuda.memory_allocated()/1e9:.2f} GB used")
print(f"✅ model + tokenizer loaded — run remaining steps in order.")

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


✅ Ready — GPU: Tesla T4
✅ VRAM : 3.51 GB used
✅ model + tokenizer loaded — run remaining steps in order.


In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError('❌ No GPU detected. Go to Runtime → Change runtime type → T4 GPU')
print(result.stdout)
print('✅ GPU ready.')

Wed Jun 17 11:31:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P8              8W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 📦 Step 1 — Install Dependencies

In [ ]:
# ── Step 1: Install dependencies (Colab-safe, June 2025) ──────────────────────

# First, upgrade pip itself to avoid resolver issues
!pip install -q --upgrade pip

# Install without pinning trl/transformers/peft/accelerate so pip finds a
# compatible set automatically. Only bitsandbytes needs a minimum floor.
!pip install -q \
    "transformers>=4.45.0" \
    "datasets>=2.20.0" \
    "peft>=0.13.0" \
    "trl>=0.12.0" \
    "accelerate>=0.34.0" \
    "bitsandbytes>=0.43.0" \
    sentencepiece \
    protobuf \
    einops

# ── CRITICAL: Restart runtime after this cell ─────────────────────────────────
# Colab pre-installs old versions of these libraries. The new versions only
# take effect after a runtime restart. Run this cell, then go to:
# Runtime → Restart session → then continue from Step 2 onward.

print("✅ Packages installed.")
print("⚠️  NOW GO TO: Runtime → Restart session")
print("   Then run all cells from Step 2 downward (skip this cell).")

✅ Packages installed.
⚠️  NOW GO TO: Runtime → Restart session
   Then run all cells from Step 2 downward (skip this cell).


## 🔑 Step 2 — HuggingFace Login (Optional)
Only needed if you want to push the adapter to the Hub. Skip if not needed.

In [ ]:
# OPTIONAL: Uncomment to log in and push your adapter to HuggingFace Hub
# from huggingface_hub import login
# login()  # Will prompt for your HF token
print('⏭️  HuggingFace login skipped (local save only).')

## 📂 Step 3 — Upload & Load the Dataset
Upload your `scam_dataset.jsonl` file when prompted.

In [ ]:
import json
from google.colab import files

# ── Upload JSONL ──────────────────────────────────────────────────────────────
print('📤 Please upload your scam_dataset.jsonl file...')
uploaded = files.upload()

# Find the uploaded file
jsonl_filename = [k for k in uploaded.keys() if k.endswith('.jsonl')]
if not jsonl_filename:
    raise FileNotFoundError('❌ No .jsonl file found in upload. Please upload scam_dataset.jsonl')
DATASET_PATH = jsonl_filename[0]
print(f'✅ Loaded: {DATASET_PATH}')

📤 Please upload your scam_dataset.jsonl file...


Saving scam_dataset.jsonl to scam_dataset (1).jsonl
✅ Loaded: scam_dataset (1).jsonl


In [ ]:
# ── Validate every row ────────────────────────────────────────────────────────
REQUIRED_KEYS  = {'instruction', 'input', 'output'}
OUTPUT_FIELDS  = {'safety_score', 'label', 'scam_type',
                  'tactics', 'smart_questions', 'emotional_reassurance', 'advice'}

rows, errors = [], []

with open(DATASET_PATH, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f, 1):
        line = line.strip()
        if not line:
            continue
        try:
            obj = json.loads(line)
        except json.JSONDecodeError as e:
            errors.append(f'Row {i}: JSON parse error — {e}'); continue

        missing = REQUIRED_KEYS - obj.keys()
        if missing:
            errors.append(f'Row {i}: Missing top-level keys {missing}'); continue

        # Parse the output JSON block (after </think> tag)
        raw_out = obj['output']
        if '</think>' in raw_out:
            json_part = raw_out.split('</think>')[1].strip()
        else:
            json_part = raw_out.strip()
        try:
            parsed_out = json.loads(json_part)
        except json.JSONDecodeError as e:
            errors.append(f'Row {i}: Output JSON parse error — {e}'); continue

        missing_out = OUTPUT_FIELDS - parsed_out.keys()
        if missing_out:
            errors.append(f'Row {i}: Output missing fields {missing_out}'); continue

        rows.append(obj)

# ── Report ────────────────────────────────────────────────────────────────────
if errors:
    print(f'⚠️  {len(errors)} validation error(s):')
    for e in errors:
        print(f'   {e}')
print(f'\n✅ Valid rows: {len(rows)}')

# Label distribution
from collections import Counter
dist = Counter()
for r in rows:
    raw = r['output']
    jp  = raw.split('</think>')[1].strip() if '</think>' in raw else raw
    dist[json.loads(jp)['scam_type']] += 1
print('\n📊 Distribution:')
for k, v in sorted(dist.items()):
    print(f'   {k:35s} → {v}')


✅ Valid rows: 40

📊 Distribution:
   bank_fraud                          → 10
   government_impersonation            → 10
   job_scam                            → 15
   safe_call                           → 5


## 🔤 Step 4 — Prompt Formatter
Converts each JSONL row into the **ChatML format** that `Qwen2.5-Instruct` expects.

In [ ]:
# ── System prompt ─────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """You are ScamShield, an expert AI system for detecting phone call scams.

When given a phone call transcript you MUST:
1. Reason step-by-step inside <think>...</think> tags.
2. Return ONLY a valid JSON object (no markdown, no extra text) with exactly these fields:
   - safety_score   : integer 0-100 (0=certain scam, 100=definitely safe)
   - label          : "SCAM" or "SAFE"
   - scam_type      : one of [job_scam, bank_fraud, government_impersonation, safe_call]
   - tactics        : list of strings (empty list if SAFE)
   - smart_questions : list of exactly 3 verification questions
   - emotional_reassurance : string — a calming message for the user
   - advice         : string — clear actionable next steps"""


def format_prompt(row: dict) -> str:
    """Format a dataset row into Qwen2.5 ChatML format."""
    return (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n"
        f"{row['instruction']}\n\n"
        f"{row['input']}<|im_end|>\n"
        f"<|im_start|>assistant\n"
        f"{row['output']}<|im_end|>"
    )


# Preview one formatted example
sample = format_prompt(rows[0])
print('── Formatted prompt preview (first 800 chars) ──')
print(sample[:800])
print('...')
print(f'\n✅ Total characters in sample: {len(sample)}')

── Formatted prompt preview (first 800 chars) ──
<|im_start|>system
You are ScamShield, an expert AI system for detecting phone call scams.

When given a phone call transcript you MUST:
1. Reason step-by-step inside <think>...</think> tags.
2. Return ONLY a valid JSON object (no markdown, no extra text) with exactly these fields:
   - safety_score   : integer 0-100 (0=certain scam, 100=definitely safe)
   - label          : "SCAM" or "SAFE"
   - scam_type      : one of [job_scam, bank_fraud, government_impersonation, safe_call]
   - tactics        : list of strings (empty list if SAFE)
   - smart_questions : list of exactly 3 verification questions
   - emotional_reassurance : string — a calming message for the user
   - advice         : string — clear actionable next steps<|im_end|>
<|im_start|>user
Analyze the phone call transcript for
...

✅ Total characters in sample: 2743


In [ ]:
# ── Build HuggingFace Dataset ─────────────────────────────────────────────────
from datasets import Dataset

formatted = [{'text': format_prompt(r)} for r in rows]
hf_dataset = Dataset.from_list(formatted)

# 90/10 train-eval split
split = hf_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split['train']
eval_dataset  = split['test']

print(f'✅ Train samples : {len(train_dataset)}')
print(f'✅ Eval  samples : {len(eval_dataset)}')

✅ Train samples : 36
✅ Eval  samples : 4


In [ ]:
# ── Diagnostic: confirm everything is ready before loading the model ──────────

import torch
import bitsandbytes as bnb
from transformers import BitsAndBytesConfig

print("=== Pre-load environment check ===")
print(f"CUDA available      : {torch.cuda.is_available()}")
print(f"GPU                 : {torch.cuda.get_device_name(0)}")
print(f"VRAM free           : {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")
print(f"VRAM total          : {torch.cuda.mem_get_info()[1]/1e9:.1f} GB")
print(f"bitsandbytes        : {bnb.__version__}")
print(f"PyTorch             : {torch.__version__}")
print(f"CUDA version        : {torch.version.cuda}")
print("=== All good — safe to run Step 5 ===")

=== Pre-load environment check ===
CUDA available      : True
GPU                 : Tesla T4
VRAM free           : 15.5 GB
VRAM total          : 15.6 GB
bitsandbytes        : 0.49.2
PyTorch             : 2.11.0+cu128
CUDA version        : 12.8
=== All good — safe to run Step 5 ===


## 🤖 Step 5 — Load Model with 4-bit Quantization (QLoRA)

In [ ]:
# ── Quick fix: hf_device_map was renamed in newer transformers ────────────────

import torch

vram_used  = torch.cuda.memory_allocated() / 1e9
vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"📊 VRAM used  : {vram_used:.2f} GB / {vram_total:.1f} GB")

# hf_device_map was removed in newer transformers — use this instead
device_map = getattr(model, 'hf_device_map', None) or {
    k: str(v.device) for k, v in model.named_parameters()
    if hasattr(v, 'device')
}
print(f"   Model device : {next(model.parameters()).device}")
print(f"   Model dtype  : {next(model.parameters()).dtype}")
print(f"   Model type   : {type(model).__name__}")

# Confirm model + tokenizer both exist before Step 6
assert model is not None, "model is None"
assert tokenizer is not None, "tokenizer is None"

print("\n✅ model and tokenizer confirmed — safe to proceed to Step 6.")

📊 VRAM used  : 2.06 GB / 15.6 GB
   Model device : cuda:0
   Model dtype  : torch.bfloat16
   Model type   : Qwen2ForCausalLM

✅ model and tokenizer confirmed — safe to proceed to Step 6.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 1B — Post-restart verification (run this AFTER restarting session)
# ═══════════════════════════════════════════════════════════════════════════════

import torch
import bitsandbytes as bnb
from transformers import BitsAndBytesConfig
import transformers, peft, trl, accelerate, datasets

print("📋 Environment check:")
print(f"   Python              : {__import__('sys').version.split()[0]}")
print(f"   PyTorch             : {torch.__version__}")
print(f"   CUDA available      : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU                 : {torch.cuda.get_device_name(0)}")
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"   VRAM                : {mem_gb:.1f} GB")
print(f"   transformers        : {transformers.__version__}")
print(f"   bitsandbytes        : {bnb.__version__}")
print(f"   peft                : {peft.__version__}")
print(f"   trl                 : {trl.__version__}")
print(f"   accelerate          : {accelerate.__version__}")
print(f"   datasets            : {datasets.__version__}")

# Prove 4-bit quantization actually works with a tiny test
print("\n🧪 Testing 4-bit quantization end-to-end...")
try:
    import torch.nn as nn

    # Create a tiny linear layer and quantize it to 4-bit
    test_layer = nn.Linear(64, 64).cuda()
    test_input = torch.randn(1, 64, device='cuda', dtype=torch.float16)

    # This uses the same code path as model loading
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    print("   BitsAndBytesConfig  : ✅ created successfully")

    # Test that the Linear4bit class is accessible (used during model load)
    test_4bit = bnb.nn.Linear4bit(64, 64, bias=False).cuda()
    print("   Linear4bit layer    : ✅ created successfully")

    print("\n✅ All checks passed — safe to continue to Step 2")

except Exception as e:
    print(f"\n❌ 4-bit test failed: {e}")
    print("\n🔧 Recovery steps:")
    print("   1. Runtime → Factory reset runtime")
    print("   2. Runtime → Change runtime type → T4 GPU")
    print("   3. Run the install cell again from scratch")

📋 Environment check:
   Python              : 3.12.13
   PyTorch             : 2.11.0+cu128
   CUDA available      : True
   GPU                 : Tesla T4
   VRAM                : 15.6 GB
   transformers        : 5.12.1
   bitsandbytes        : 0.49.2
   peft                : 0.19.1
   trl                 : 1.6.0
   accelerate          : 1.14.0
   datasets            : 5.0.0

🧪 Testing 4-bit quantization end-to-end...
   BitsAndBytesConfig  : ✅ created successfully
   Linear4bit layer    : ✅ created successfully

✅ All checks passed — safe to continue to Step 2


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [ ]:
# ── Re-install missing packages (run once, no restart needed) ─────────────────
!pip install -q "peft>=0.13.0" "trl>=0.12.0" "accelerate>=0.34.0"

# Verify
import peft, trl, accelerate
print(f"✅ peft       : {peft.__version__}")
print(f"✅ trl        : {trl.__version__}")
print(f"✅ accelerate : {accelerate.__version__}")
print("\n▶️  Now re-run Step 6.")

✅ peft       : 0.19.1
✅ trl        : 1.6.0
✅ accelerate : 1.14.0

▶️  Now re-run Step 6.


## 🔧 Step 6 — LoRA Configuration (PEFT)

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

# ── Prepare for k-bit training ────────────────────────────────────────────────
# Enables gradient checkpointing and casts non-quantized params to fp32
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)

# ── LoRA Config ───────────────────────────────────────────────────────────────
# Target the attention and MLP projection layers in Qwen2.5
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,

    # ── Rank & scale ──────────────────────────────────────────────────────────
    r=16,               # LoRA rank — 16 is a good balance for 3B on T4
    lora_alpha=32,      # Alpha = 2×r is the standard starting point
    lora_dropout=0.05,  # Small dropout to prevent adapter overfitting

    # ── Target modules for Qwen2.5 architecture ───────────────────────────────
    # These are the linear projections inside each transformer block
    target_modules=[
        'q_proj',    # Query projection
        'k_proj',    # Key projection
        'v_proj',    # Value projection
        'o_proj',    # Output projection
        'gate_proj', # MLP gate (SwiGLU)
        'up_proj',   # MLP up
        'down_proj', # MLP down
    ],

    bias='none',                    # Don't adapt bias terms
    inference_mode=False,
)

model = get_peft_model(model, lora_config)

# ── Parameter summary ─────────────────────────────────────────────────────────
model.print_trainable_parameters()
print('\n✅ LoRA adapters attached.')

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607

✅ LoRA adapters attached.


In [ ]:
# ── Rebuild datasets (run if train_dataset is undefined) ──────────────────────
import json
from datasets import Dataset

# ── 1. Load your JSONL file ───────────────────────────────────────────────────
JSONL_PATH = "scam_dataset.jsonl"   # adjust path if needed

rows = []
with open(JSONL_PATH, "r") as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))

print(f"   Loaded {len(rows)} rows from {JSONL_PATH}")

# ── 2. Format each row into Qwen2.5 ChatML format ────────────────────────────
def format_row(row):
    return {
        "text": (
            f"<|im_start|>system\n"
            f"You are a scam detection AI. Analyze phone call transcripts and return structured JSON assessments.\n"
            f"<|im_end|>\n"
            f"<|im_start|>user\n"
            f"{row['instruction']}\n\n{row['input']}\n"
            f"<|im_end|>\n"
            f"<|im_start|>assistant\n"
            f"{row['output']}\n"
            f"<|im_end|>"
        )
    }

formatted = [format_row(r) for r in rows]

# ── 3. Split 90/10 train/eval ─────────────────────────────────────────────────
split_idx = max(1, int(len(formatted) * 0.9))
train_dataset = Dataset.from_list(formatted[:split_idx])
eval_dataset  = Dataset.from_list(formatted[split_idx:])

print(f"   Train samples : {len(train_dataset)}")
print(f"   Eval samples  : {len(eval_dataset)}")
print("✅ Datasets ready — re-run Step 7 now.")

   Loaded 40 rows from scam_dataset.jsonl
   Train samples : 36
   Eval samples  : 4
✅ Datasets ready — re-run Step 7 now.


In [ ]:
# ── Run this diagnostic first to see what SFTConfig actually accepts ──────────
from trl import SFTConfig
import inspect

params = inspect.signature(SFTConfig.__init__).parameters
print("SFTConfig accepts these non-standard params:")
standard = {'self','kwargs','args'}
for p in params:
    if p not in standard:
        print(f"  {p}")

SFTConfig accepts these non-standard params:
  output_dir
  per_device_train_batch_size
  num_train_epochs
  max_steps
  learning_rate
  lr_scheduler_type
  lr_scheduler_kwargs
  warmup_steps
  optim
  optim_args
  weight_decay
  adam_beta1
  adam_beta2
  adam_epsilon
  optim_target_modules
  gradient_accumulation_steps
  average_tokens_across_devices
  max_grad_norm
  label_smoothing_factor
  bf16
  fp16
  bf16_full_eval
  fp16_full_eval
  tf32
  gradient_checkpointing
  gradient_checkpointing_kwargs
  torch_compile
  torch_compile_backend
  torch_compile_mode
  use_liger_kernel
  liger_kernel_config
  use_cache
  neftune_noise_alpha
  torch_empty_cache_steps
  auto_find_batch_size
  logging_strategy
  logging_steps
  logging_first_step
  log_on_each_node
  logging_nan_inf_filter
  include_num_input_tokens_seen
  log_level
  log_level_replica
  disable_tqdm
  report_to
  run_name
  project
  trackio_space_id
  trackio_bucket_id
  trackio_static_space_id
  eval_strategy
  eval_steps
  

In [ ]:
# ── Reattach LoRA adapters (run if you see the quantized model error) ─────────
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    inference_mode=False,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("\n✅ LoRA adapters attached — re-run Step 7 now.")

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607

✅ LoRA adapters attached — re-run Step 7 now.


## 🏋️ Step 7 — Training Configuration & SFTTrainer

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 7 — Training Configuration & SFTTrainer (TRL 1.6.0 exact)
# ═══════════════════════════════════════════════════════════════════════════════

import os
from trl import SFTConfig, SFTTrainer

OUTPUT_DIR = "./scam_shield_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,

    # ── Epochs & batch size ───────────────────────────────────────────────────
    num_train_epochs=4,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,

    # ── Learning rate ─────────────────────────────────────────────────────────
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,

    # ── Memory optimizations ──────────────────────────────────────────────────
    fp16=False,
    bf16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    dataloader_pin_memory=False,

    # ── Logging & saving ──────────────────────────────────────────────────────
    logging_steps=5,
    save_strategy="epoch",
    eval_strategy="epoch",
    load_best_model_at_end=False,
    report_to="none",

    # ── Sequence & dataset config (TRL 1.6 names) ─────────────────────────────
    max_length=1024,                     # ← was max_seq_length, now max_length
    dataset_text_field="text",
    loss_type="nll",                     # ← pin to silence FutureWarning
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

print(f"✅ SFTTrainer initialized (trl {__import__('trl').__version__})")
print(f"   Train samples : {len(train_dataset)}")
print(f"   Eval samples  : {len(eval_dataset)}")
print("   Ready for Step 8 → trainer.train()")

Adding EOS to train dataset:   0%|          | 0/36 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/36 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

✅ SFTTrainer initialized (trl 1.6.0)
   Train samples : 36
   Eval samples  : 4
   Ready for Step 8 → trainer.train()


## 🚀 Step 8 — Train!

In [ ]:
import time

print('🚀 Starting fine-tuning...')
print('   This will take ~15-25 minutes on a T4 GPU.\n')

start = time.time()
train_result = trainer.train()
elapsed = time.time() - start

print(f'\n✅ Training complete in {elapsed/60:.1f} minutes.')
print(f'   Final train loss : {train_result.training_loss:.4f}')

# Log metrics
trainer.log_metrics('train', train_result.metrics)
trainer.save_metrics('train', train_result.metrics)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


🚀 Starting fine-tuning...
   This will take ~15-25 minutes on a T4 GPU.



Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,3.008475,2.560295,1.784188,0.487151,15714.000000
2,2.338000,1.989636,2.169180,0.545107,31428.000000
3,1.797741,1.610288,1.759071,0.604702,47142.000000
4,1.495703,1.541761,1.585035,0.615090,62856.000000



✅ Training complete in 11.5 minutes.
   Final train loss : 2.1600
***** train metrics *****
  epoch                    =        4.0
  total_flos               =   985112GF
  train_loss               =       2.16
  train_runtime            = 0:11:32.13
  train_samples_per_second =      0.208
  train_steps_per_second   =      0.029


## 💾 Step 9 — Save LoRA Adapter Only
We save only the adapter weights (~50 MB) — NOT the full 3B model.

In [ ]:
ADAPTER_SAVE_PATH = './scam_shield_adapter'

# Save adapter weights + config
trainer.model.save_pretrained(ADAPTER_SAVE_PATH)
tokenizer.save_pretrained(ADAPTER_SAVE_PATH)

# Check what was saved
import os
saved_files = os.listdir(ADAPTER_SAVE_PATH)
print(f'✅ Adapter saved to: {ADAPTER_SAVE_PATH}')
print('   Files saved:')
for f in saved_files:
    size = os.path.getsize(os.path.join(ADAPTER_SAVE_PATH, f)) / 1e6
    print(f'   {f:40s}  {size:.2f} MB')

✅ Adapter saved to: ./scam_shield_adapter
   Files saved:
   adapter_config.json                       0.00 MB
   tokenizer.json                            11.42 MB
   chat_template.jinja                       0.00 MB
   tokenizer_config.json                     0.00 MB
   README.md                                 0.01 MB
   adapter_model.safetensors                 59.93 MB


In [ ]:
# ── Zip & download adapter ────────────────────────────────────────────────────
import shutil
from google.colab import files

zip_path = './scam_shield_adapter.zip'
shutil.make_archive('./scam_shield_adapter', 'zip', ADAPTER_SAVE_PATH)

zip_size = os.path.getsize(zip_path) / 1e6
print(f'📦 Adapter zipped: {zip_path} ({zip_size:.1f} MB)')

files.download(zip_path)
print('⬇️  Download started.')

📦 Adapter zipped: ./scam_shield_adapter.zip (48.7 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  Download started.


## 🔍 Step 10 — Inference
Load the fine-tuned adapter and test it on new call transcripts.

In [ ]:
import json
import torch
import re
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID         = 'Qwen/Qwen2.5-3B-Instruct'
ADAPTER_PATH     = './scam_shield_adapter'   # Path to saved adapter

SYSTEM_PROMPT = """You are ScamShield, an expert AI system for detecting phone call scams.

When given a phone call transcript you MUST:
1. Reason step-by-step inside <think>...</think> tags.
2. Return ONLY a valid JSON object (no markdown, no extra text) with exactly these fields:
   - safety_score   : integer 0-100 (0=certain scam, 100=definitely safe)
   - label          : "SCAM" or "SAFE"
   - scam_type      : one of [job_scam, bank_fraud, government_impersonation, safe_call]
   - tactics        : list of strings (empty list if SAFE)
   - smart_questions : list of exactly 3 verification questions
   - emotional_reassurance : string — a calming message for the user
   - advice         : string — clear actionable next steps"""

# ── Load adapter on top of base model ─────────────────────────────────────────
print('Loading base model + adapter for inference...')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

inf_tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH, trust_remote_code=True)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)

inf_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
inf_model.eval()

print('✅ Model + adapter loaded for inference.')

Loading base model + adapter for inference...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


✅ Model + adapter loaded for inference.


In [ ]:
# ── Inference function ────────────────────────────────────────────────────────

def analyze_call(transcript: str, max_new_tokens: int = 800) -> dict:
    """
    Run scam detection on a phone call transcript.

    Args:
        transcript: The text of the phone call (from Whisper or direct input)
        max_new_tokens: Max output tokens (increase for longer reasoning)

    Returns:
        Parsed JSON dict with scam assessment fields
    """
    prompt = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n"
        f"Analyze the phone call transcript for fraud and return a structured assessment.\n\n"
        f"Caller: {transcript}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

    inputs = inf_tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1800)
    inputs = {k: v.to(inf_model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = inf_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,         # Low temperature = deterministic structured output
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
            pad_token_id=inf_tokenizer.eos_token_id,
            eos_token_id=inf_tokenizer.encode('<|im_end|>')[0],
        )

    # Decode only the newly generated tokens
    new_tokens = output_ids[0][inputs['input_ids'].shape[1]:]
    raw_output = inf_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # ── Parse output ──────────────────────────────────────────────────────────
    # Strip <think>...</think> block to isolate the JSON
    if '</think>' in raw_output:
        think_block = raw_output.split('</think>')[0].replace('<think>', '').strip()
        json_part   = raw_output.split('</think>')[1].strip()
    else:
        think_block = ''
        json_part   = raw_output

    # Clean any stray markdown fences
    json_part = re.sub(r'^```json|^```|```$', '', json_part, flags=re.MULTILINE).strip()

    try:
        result = json.loads(json_part)
    except json.JSONDecodeError:
        # Fallback: return raw for debugging
        result = {'parse_error': True, 'raw_output': raw_output}

    result['_reasoning'] = think_block  # Attach reasoning for transparency
    return result


print('✅ analyze_call() ready.')

✅ analyze_call() ready.


In [33]:
# ── Test Case 1: Job Scam ─────────────────────────────────────────────────────
test_transcript_1 = ("")

print('📞 Analyzing transcript 1 (Job Scam)...\n')
result1 = analyze_call(test_transcript_1)

print('── REASONING ──────────────────────────────────────────────────────')
print(result1.get('_reasoning', 'N/A')[:400])
print('\n── ASSESSMENT ─────────────────────────────────────────────────────')
display_result = {k: v for k, v in result1.items() if not k.startswith('_')}
print(json.dumps(display_result, indent=2))

📞 Analyzing transcript 1 (Job Scam)...



KeyboardInterrupt: 

In [34]:
test_transcript_1 = ("""
John: Hi, is this Bob? I'm John Miller, Senior Compliance Director with the Regional Housing Board. Your recent eviction notice on Elm Street was flagged on our system this morning.

Victim: Hello? Yes, this is Bob. I don’t understand. The landlord told me I have to leave in three weeks because he is selling the house. I don't know where to go.

John: That is why we intervened, Bob. Under Section 8-B of the Emergency Housing Act, we have an allocated, fully furnished 2-bedroom townhouse reserved for you ten minutes away. The government-subsidized rate is a flat $1,000 a month with all utilities free. Because it is a pre-approved grant, you don't need to fill out any background check forms.

Victim: $1,000 with free utilities? Are you sure? I can't read those big lease papers, John. Everywhere else asks me to fill out forms I don't understand.

John: You don't have to read a single line, Bob. We handle all waivers digitally on our backend. To finalize the transfer and get your digital keys by tomorrow morning, the system just requires a flat $1,000 holding deposit. It goes into a secure state-insured escrow and counts as your first month's rent.

Victim: Tomorrow morning? That would be a huge blessing. But can you meet me at the house tonight? I want to see it before I hand over my money.

John: The agent is locked out until tomorrow's check at 9:00 AM. But here is the problem, Bob: our dashboard shows three other displaced families trying to get this exact unit. The system automatically routes it to whoever pays the deposit first. To secure it, you must send the $1,000 via Zelle or Apple Pay within the hour.

Victim: Within the hour? John, $1,000 is all the cash I have left to my name. I can't just send all my money through the phone to someone I haven't met.

John: I respect your caution, but this is a secure government procedure. I am texting you an official video walkthrough right now so you don't have to read anything. If you view the property tomorrow and don't like it, we hit a manual override button and issue an instantaneous refund directly back to your phone. It is 100% risk-protected.

Victim: You sound official, John, and I really need help. But my brother helps me with my banking since I can't read. Let me wait until he gets off work. Can you hold the house for me until 4:00 PM today?

John: The system automatically locks the terminal after 60 minutes to prevent favoritism, Bob. If the deposit isn't in by 1:00 PM sharp, the server automatically passes the townhouse to the next family on the waiting list. That is the absolute limit of my security clearance.

Victim: Okay. Send the video. I will wait for my brother to call me, and if we can figure out the phone banking, I will call you back before 1:00 PM.

John: Understood, Bob. Check your texts in two minutes for the link. Let's make sure we secure this home for you today. Goodbye.

Victim: Thank you, John. Bye.
""")

print('📞 Analyzing transcript')
result1 = analyze_call(test_transcript_1)

print('── REASONING ──────────────────────────────────────────────────────')
print(result1.get('_reasoning', 'N/A')[:400])

print('\n── ASSESSMENT ─────────────────────────────────────────────────────')
display_result = {k: v for k, v in result1.items() if not k.startswith('_')}
print(json.dumps(display_result, indent=2))

📞 Analyzing transcript
── REASONING ──────────────────────────────────────────────────────
This is a classic government impersonation scam. The caller pretends to be a government agency representative who has 'flagged' an eviction notice. They offer a subsidized housing unit that sounds too good to be true, complete with free utilities and a flat monthly rent. The key tactic is to create urgency around the 'government intervention' while simultaneously using the 'secure escrow' payment 

── ASSESSMENT ─────────────────────────────────────────────────────
{
  "parse_error": true,
  "raw_output": "<think>\nThis is a classic government impersonation scam. The caller pretends to be a government agency representative who has 'flagged' an eviction notice. They offer a subsidized housing unit that sounds too good to be true, complete with free utilities and a flat monthly rent. The key tactic is to create urgency around the 'government intervention' while simultaneously using the 'secure escr

In [ ]:
# ── Test Case 2: Bank Fraud ───────────────────────────────────────────────────
test_transcript_2 = (
    "Good evening, this is the fraud prevention unit at Barclays Bank. "
    "We've detected 4 suspicious transactions on your account totalling "
    "£2,100. To reverse these and secure your account, I need you to "
    "confirm your card number, sort code, and the one-time passcode we just "
    "texted to your mobile."
)

print('📞 Analyzing transcript 2 (Bank Fraud)...\n')
result2 = analyze_call(test_transcript_2)

print('── ASSESSMENT ─────────────────────────────────────────────────────')
display_result2 = {k: v for k, v in result2.items() if not k.startswith('_')}
print(json.dumps(display_result2, indent=2))

📞 Analyzing transcript 2 (Bank Fraud)...

── ASSESSMENT ─────────────────────────────────────────────────────
{
  "safety_score": 5,
  "label": "SCAM",
  "scam_type": "bank_fraud",
  "tactics": [
    "text-based PIN request",
    "emotional urgency",
    "real transaction data"
  ],
  "smart_questions": [
    "What is my actual Barclays security PIN?",
    "Can I call Barclays back immediately to verify this?",
    "Why did you not send me a formal letter instead?"
  ],
  "emotional_reassurance": "It\u2019s okay to feel scared right now. You\u2019re doing the right thing by hanging up and calling Barclays yourself.",
  "advice": "Do not share any card details or PIN numbers over the phone. Call Barclays directly using their official contact number to report this."
}


In [ ]:
# ── Test Case 3: Safe Call ────────────────────────────────────────────────────
test_transcript_3 = (
    "Hi, this is Amara calling from Mountain View Medical Centre. "
    "I'm just reminding you about your appointment with Dr. Osei this "
    "Thursday at 10:30am. Please bring your insurance card and arrive "
    "10 minutes early. Call us back at the number on your appointment "
    "card if you need to reschedule. Thank you!"
)

print('📞 Analyzing transcript 3 (Safe Call)...\n')
result3 = analyze_call(test_transcript_3)

print('── ASSESSMENT ─────────────────────────────────────────────────────')
display_result3 = {k: v for k, v in result3.items() if not k.startswith('_')}
print(json.dumps(display_result3, indent=2))

📞 Analyzing transcript 3 (Safe Call)...

── ASSESSMENT ─────────────────────────────────────────────────────
{
  "safety_score": 98,
  "label": "SAFE",
  "scam_type": "safe_call",
  "tactics": [],
  "smart_questions": [
    "Can you please check your appointment card now to verify the details?",
    "Is there any other way to reach Dr. Osei besides calling our clinic directly?",
    "Could you tell me what day and time your appointment was originally scheduled?"
  ],
  "emotional_reassurance": "It\u2019s great that someone is reaching out to remind you about your important medical appointment. You\u2019re doing the right thing by verifying the information yourself.",
  "advice": "Call the number on your appointment card to double-check all the details before arriving."
}


## 📊 Step 11 — Evaluation Summary

In [ ]:
# ── Run on all eval rows and measure JSON parse rate ──────────────────────────
from tqdm import tqdm

print(f'🔍 Evaluating on {len(eval_dataset)} held-out samples...\n')

correct_label  = 0
correct_type   = 0
json_ok        = 0
total          = 0

for sample in tqdm(eval_dataset):
    # Extract ground truth from the formatted text
    raw_text = sample['text']
    gt_block = raw_text.split('<|im_start|>assistant\n')[1].replace('<|im_end|>', '').strip()
    if '</think>' in gt_block:
        gt_json_str = gt_block.split('</think>')[1].strip()
    else:
        gt_json_str = gt_block

    try:
        gt = json.loads(gt_json_str)
    except:
        continue

    # Get transcript from text block
    try:
        transcript = raw_text.split('Caller: ')[1].split('<|im_end|>')[0].strip()
    except:
        continue

    pred = analyze_call(transcript, max_new_tokens=600)
    total += 1

    if 'parse_error' not in pred:
        json_ok += 1
        if pred.get('label')     == gt.get('label')     : correct_label += 1
        if pred.get('scam_type') == gt.get('scam_type') : correct_type  += 1

print(f'\n{'─'*45}')
print(f'  Total evaluated     : {total}')
print(f'  JSON parse rate     : {json_ok}/{total} ({100*json_ok/max(total,1):.0f}%)')
print(f'  Label accuracy      : {correct_label}/{json_ok} ({100*correct_label/max(json_ok,1):.0f}%)')
print(f'  Scam type accuracy  : {correct_type}/{json_ok} ({100*correct_type/max(json_ok,1):.0f}%)')
print(f'{'─'*45}')

🔍 Evaluating on 4 held-out samples...



100%|██████████| 4/4 [02:56<00:00, 44.21s/it]


─────────────────────────────────────────────
  Total evaluated     : 4
  JSON parse rate     : 4/4 (100%)
  Label accuracy      : 4/4 (100%)
  Scam type accuracy  : 2/4 (50%)
─────────────────────────────────────────────


## 🚀 (Optional) Step 12 — Push Adapter to HuggingFace Hub

In [ ]:
# OPTIONAL: Push to HF Hub
# Uncomment and fill in your username and repo name

# HF_USERNAME = 'your-username'
# REPO_NAME   = 'scam-shield-qwen2.5-3b-lora'
#
# from huggingface_hub import login
# login()
#
# inf_model.push_to_hub(f'{HF_USERNAME}/{REPO_NAME}')
# inf_tokenizer.push_to_hub(f'{HF_USERNAME}/{REPO_NAME}')
# print(f'✅ Adapter pushed to: https://huggingface.co/{HF_USERNAME}/{REPO_NAME}')

print('⏭️  Hub push skipped. Uncomment lines above to enable.')

---
## 📋 Integration Notes for Your Backend

### How to integrate with Whisper
```python
# In your backend pipeline:
import whisper

# Step 1: Transcribe audio
whisper_model = whisper.load_model('base')
transcript_result = whisper_model.transcribe('call_recording.mp3')
transcript_text = transcript_result['text']

# Step 2: Analyze with your fine-tuned ScamShield model
assessment = analyze_call(transcript_text)

# Step 3: Use the structured output
if assessment['label'] == 'SCAM':
    print(f"⚠️  SCAM DETECTED (confidence: {100 - assessment['safety_score']}%)")
    print(f"Type    : {assessment['scam_type']}")
    print(f"Tactics : {assessment['tactics']}")
    print(f"Ask the caller: {assessment['smart_questions'][0]}")
```

### Adapter loading in production (no retraining)
```python
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

base = AutoModelForCausalLM.from_pretrained(
    'Qwen/Qwen2.5-3B-Instruct',
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                           bnb_4bit_compute_dtype=torch.bfloat16),
    device_map='auto', trust_remote_code=True,
)
model = PeftModel.from_pretrained(base, './scam_shield_adapter')
tokenizer = AutoTokenizer.from_pretrained('./scam_shield_adapter')
```

### To expand the dataset (recommended)
- Add more rows to `scam_dataset.jsonl` and re-run from Step 3
- Target **200–500 rows** for production-grade accuracy
- Include local scam variants relevant to your target region
- Add more `SAFE` examples for better precision (reduce false positives)

### Recommended next steps
1. Expand dataset to 200+ rows
2. Run 3-fold cross-validation
3. Add a confidence threshold (e.g., `safety_score < 30` → auto-flag)
4. Connect `analyze_call()` directly to your Whisper output stream